# Machine Learnig in Raw Data 2

In [ ]:
# -*- coding: utf-8 -*-
# ====================================================================================
# COMPLETE PIPELINE FOR TRAINING, SAVING, AND RENDERING FROM FILES (without retraining)
# ====================================================================================

#A: Δt=0.01 s
#B: Δt=0.02 s
#C: Δt=0.05 s
#D: Δt=0.15 s
#E: Δt=0.25 s
#F: Δt=0.50 s
#G: Δt=1.00 s
#H: Δt=2.00 s

#A: user 0007
#B: user 0008
#C: user 0010

from pathlib import Path
import sys
import re
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc
)

#---------- Reproducibility Configuration
sys.path.insert(0, str(Path("src").resolve()))
from reproducibility import DEFAULT_SEED, configure_reproducibility
from plotting import (
    build_by_user_vector_collages_for_stem,
    build_overall_vector_collages,
    draw_confusion_matrix,
)

SEED = DEFAULT_SEED
configure_reproducibility(SEED, include_tensorflow=True)
print(f"Global reproducibility seed configured: {SEED}")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# ---------------- Figure style (publication quality, no seaborn)
mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.titlelocation": "left",
    "axes.titleweight": "regular",
    "axes.grid": False,
    "figure.autolayout": False,     # usamos tight_layout / subplots_adjust manual
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif", "Nimbus Roman"]

# ---------------- Paths
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "Data/3-Data-ML-1"
RESULTS_DIR = BASE_DIR / "Outputs/Results-Data-ML-2-v4"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Collage directories
OVERALL_COLLAGES_DIR = RESULTS_DIR / "_collages_overall"
OVERALL_COLLAGES_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Datasets 
WINDOWS = [0.01, 0.02, 0.05, 0.15, 0.25, 0.50, 1.00, 2.00]
STEMS   = [f"ml_win_{w:.2f}s" for w in WINDOWS]
CSV_PATHS = {stem: DATA_DIR / f"{stem}.csv" for stem in STEMS}

# ---------------- Utilidades
RANDOM_STATE = SEED
configure_reproducibility(RANDOM_STATE, include_tensorflow=True)


def kernel_seed(offset: int):
    return keras.initializers.GlorotUniform(seed=RANDOM_STATE + offset)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def _is_feature_col(c: str) -> bool:
    return bool(re.match(r"^S[1-7]_\d{5}$", str(c)))

def basic_info(df: pd.DataFrame, name: str) -> str:
    n_rows, n_cols = df.shape
    feat_cols = [c for c in df.columns if _is_feature_col(c)]
    users = df["user"].astype(str).unique().tolist()
    gestures = df["gesture"].astype(str).unique().tolist()
    info = []
    info.append(f"[{name}] shape = {n_rows} x {n_cols}")
    info.append(f"[{name}] #features = {len(feat_cols)}")
    info.append(f"[{name}] users = {users}")
    info.append(f"[{name}] gestures = {gestures}")
    vc_users = df["user"].astype(str).value_counts().sort_index()
    vc_gest  = df["gesture"].astype(str).value_counts().sort_index()
    info.append(f"[{name}] filas por user:\n{vc_users.to_string()}")
    info.append(f"[{name}] filas por gesture:\n{vc_gest.to_string()}")
    return "\n".join(info)

def _min_needed_for_test(test_size: float) -> int:
    return max(2, int(np.ceil(1.0 / float(test_size))))

def safe_stratify_vector(vec: pd.Series, test_size=0.2):
    vc = vec.value_counts()
    min_needed = _min_needed_for_test(test_size)
    if (vc < min_needed).any() or vc.shape[0] < 2:
        return None
    return vec

def make_user_gesture_str(df: pd.DataFrame) -> pd.Series:
    return df["user"].astype(str) + "||" + df["gesture"].astype(str)

def split_train_test(df: pd.DataFrame, test_size=0.2,
                     prefer_order=("user_gesture", "user", "gesture")):
    feat_cols = [c for c in df.columns if _is_feature_col(c)]
    X = df[feat_cols].to_numpy(dtype=np.float32)
    y_g = df["gesture"].astype(str).values
    users = df["user"].astype(str).values

    le = LabelEncoder()
    y_int = le.fit_transform(y_g)
    n_classes = len(le.classes_)

    candidates = []
    for how in prefer_order:
        if how == "user_gesture":
            candidates.append(("user_gesture", make_user_gesture_str(df)))
        elif how == "user":
            candidates.append(("user", pd.Series(users)))
        elif how == "gesture":
            candidates.append(("gesture", pd.Series(y_int)))

    strat = None
    strat_used = None
    for name, vec in candidates:
        s = safe_stratify_vector(pd.Series(vec), test_size=test_size)
        if s is not None:
            strat = s
            strat_used = name
            break

    Xtr, Xte, ytr, yte, users_tr, users_te = train_test_split(
        X, y_int, users, test_size=test_size, random_state=RANDOM_STATE,
        stratify=strat if strat_used else None
    )
    feat_cols = [c for c in df.columns if _is_feature_col(c)]
    return (Xtr, Xte, ytr, yte, users_tr, users_te, le, n_classes, feat_cols, strat_used)

def standardize(Xtr, Xte):
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)
    return Xtr_s, Xte_s, scaler

def build_model(input_dim: int, n_classes: int, n_sensors: int = 7) -> keras.Model:
    """
    CNN1D for segmented EMG.
    Input: flat vector ordered as [S1_1..S1_T, S2_1..S2_T, ..., S7_1..S7_T] (size = 7*T).
    """
    if input_dim % n_sensors != 0:
        raise ValueError(
            f"input_dim={input_dim} is not a multiple of n_sensors={n_sensors}. "
            "The vector must have size 7*T (concatenated S1..S7)."
        )

    T = input_dim // n_sensors  # timesteps per window
    inp = layers.Input(shape=(input_dim,), name="input_flat")

    # [7*T] -> (T, 7)
    x = layers.Reshape((n_sensors, T), name="reshape_7_T")(inp)
    x = layers.Permute((2, 1), name="permute_T_7")(x)

    # Bloques Conv1D
    x = layers.Conv1D(32, kernel_size=9, padding="same", activation="relu", kernel_initializer=kernel_seed(1), name="conv1")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool1")(x)
    x = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu", kernel_initializer=kernel_seed(2), name="conv2")(x)
    x = layers.MaxPooling1D(pool_size=2, name="pool2")(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu", kernel_initializer=kernel_seed(3), name="conv3")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(128, activation="relu", kernel_initializer=kernel_seed(4), name="fc1")(x)
    x = layers.Dropout(0.1, seed=RANDOM_STATE + 101, name="drop1")(x)
    x = layers.Dense(64, activation="relu", kernel_initializer=kernel_seed(5), name="fc2")(x)
    x = layers.Dropout(0.1, seed=RANDOM_STATE + 102, name="drop2")(x)
    x = layers.Dense(32,  activation="relu", kernel_initializer=kernel_seed(6), name="fc3")(x)
    x = layers.Dropout(0.1, seed=RANDOM_STATE + 103, name="drop3")(x)

    out = layers.Dense(n_classes, activation="softmax", kernel_initializer=kernel_seed(7), name="softmax")(x)

    model = keras.Model(inputs=inp, outputs=out, name="EMG_CNN1D_Explainable")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ----------------------- INDIVIDUAL PLOTS (NO TITLES) ------------------------

def plot_training_curves(history_df: pd.DataFrame, out_base: Path):
    # Accuracy
    plt.figure(figsize=(5,5))
    if 'accuracy' in history_df:
        plt.plot(history_df['accuracy'].values, label="Training")
    if 'val_accuracy' in history_df:
        plt.plot(history_df['val_accuracy'].values, label="Validation", linestyle="--")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(loc="best"); plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    acc_png = out_base.parent / (out_base.name + "_accuracy.png")
    acc_pdf = out_base.parent / (out_base.name + "_accuracy.pdf")
    plt.savefig(str(acc_png), dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(str(acc_pdf), dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"[SAVE] {acc_png}")
    print(f"[SAVE] {acc_pdf}")
    plt.close()

    # Loss
    plt.figure(figsize=(5,5))
    if 'loss' in history_df:
        plt.plot(history_df['loss'].values, label="Training")
    if 'val_loss' in history_df:
        plt.plot(history_df['val_loss'].values, label="Validation", linestyle="--")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(loc="best"); plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    loss_png = out_base.parent / (out_base.name + "_loss.png")
    loss_pdf = out_base.parent / (out_base.name + "_loss.pdf")
    plt.savefig(str(loss_png), dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(str(loss_pdf), dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"[SAVE] {loss_png}")
    print(f"[SAVE] {loss_pdf}")
    plt.close()

def _nice_confusion(ax, cm: np.ndarray, class_names):
    draw_confusion_matrix(ax, cm, list(class_names))

def plot_confusion_from_arrays(y_true_int, y_pred_int, class_names, out_path: Path):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true_int, y_pred_int, labels=labels)
    fig, ax = plt.subplots(figsize=(6,5))
    _nice_confusion(ax, cm, class_names)
    plt.tight_layout(pad=0.2)
    png = out_path.parent / (out_path.name + ".png")
    pdf = out_path.parent / (out_path.name + ".pdf")
    plt.savefig(str(png), dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(str(pdf), dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"[SAVE] {png}")
    print(f"[SAVE] {pdf}")
    plt.close(fig)

def plot_roc_from_arrays(y_true_int, y_prob, class_names, out_path: Path):
    n_classes = len(class_names)
    y_true_ohe = np.eye(n_classes)[y_true_int]
    fpr, tpr, roc_auc = {}, {}, {}

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_ohe[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_ohe.ravel(), y_prob.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure(figsize=(7,6))
    plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"micro (AUC={roc_auc['micro']:.3f})")
    plt.plot(fpr["macro"], tpr["macro"], linestyle="--", label=f"macro (AUC={roc_auc['macro']:.3f})")
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot([0,1], [0,1], "k:", alpha=0.5)
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right", fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout(pad=0.2)
    png = out_path.parent / (out_path.name + ".png")
    pdf = out_path.parent / (out_path.name + ".pdf")
    plt.savefig(str(png), dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.savefig(str(pdf), dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"[SAVE] {png}")
    print(f"[SAVE] {pdf}")
    plt.close()

# --------------------------- ENTRENAR Y GUARDAR (PARTE 1) --------------------------

def run_one_experiment(df: pd.DataFrame, stem: str, out_dir: Path, tag: str):
    ensure_dir(out_dir)

    plots_dir = out_dir / "plots"
    ensure_dir(plots_dir)
    arrays_dir = out_dir / "arrays"
    ensure_dir(arrays_dir)

    metrics_txt = out_dir / "metrics.txt"
    metrics_json = out_dir / "metrics.json"
    classes_json = out_dir / "classes.json"
    history_csv = out_dir / "history.csv"
    ytrue_csv = arrays_dir / "y_true.csv"
    ypred_csv = arrays_dir / "y_pred.csv"
    yprob_csv = arrays_dir / "y_prob.csv"

    curves_base = plots_dir / "training_curves"

    prefer_overall = ("user_gesture", "user", "gesture")
    prefer_byuser = ("gesture", "user") if tag.startswith("user_") else prefer_overall

    Xtr, Xte, ytr, yte, users_tr, users_te, le, n_classes, feat_cols, strat_used = \
        split_train_test(df, test_size=0.2, prefer_order=prefer_byuser)

    Xtr, Xte, scaler = standardize(Xtr, Xte)

    _print_split_sizes(stem, tag, users_tr, users_te, ytr, yte, le, strat_used)

    configure_reproducibility(RANDOM_STATE, include_tensorflow=True)
    model = build_model(input_dim=Xtr.shape[1], n_classes=n_classes)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1)
    ]

    t0_fit = time.perf_counter()
    history = model.fit(
        Xtr, ytr,
        epochs=100,
        batch_size=16,
        validation_split=0.1,
        shuffle=True,
        verbose=0,
        callbacks=callbacks
    )
    t_fit = time.perf_counter() - t0_fit
    epochs_trained = len(history.history.get("loss", []))

    hist_df = pd.DataFrame({
        "epoch": np.arange(1, epochs_trained+1),
        "accuracy": history.history.get("accuracy", [None]*epochs_trained),
        "val_accuracy": history.history.get("val_accuracy", [None]*epochs_trained),
        "loss": history.history.get("loss", [None]*epochs_trained),
        "val_loss": history.history.get("val_loss", [None]*epochs_trained),
    })
    hist_df.to_csv(history_csv, index=False)

    t0_test = time.perf_counter()
    prob_te = model.predict(Xte, verbose=0)
    t_test = time.perf_counter() - t0_test
    y_pred = np.argmax(prob_te, axis=1)

    pd.DataFrame({"y_true": yte}).to_csv(ytrue_csv, index=False)
    pd.DataFrame({"y_pred": y_pred}).to_csv(ypred_csv, index=False)
    yprob_df = pd.DataFrame(prob_te, columns=[f"p_{c}" for c in le.classes_])
    yprob_df.to_csv(yprob_csv, index=False)

    with open(classes_json, "w", encoding="utf-8") as f:
        json.dump({"classes": list(le.classes_)}, f, ensure_ascii=False, indent=2)

    acc = accuracy_score(yte, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(yte, y_pred, average="macro", zero_division=0)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(yte, y_pred, average="weighted", zero_division=0)
    cls_report = classification_report(yte, y_pred, target_names=list(le.classes_), zero_division=0)

    metrics_payload = {
        "experiment": f"{stem} — {tag}",
        "stem": stem,
        "tag": tag,
        "stratify_used": strat_used,
        "train_shape": list(Xtr.shape),
        "test_shape": list(Xte.shape),
        "timings": {"train_time_s": float(t_fit), "test_time_s": float(t_test), "epochs_trained": int(epochs_trained)},
        "metrics": {
            "accuracy": round(float(acc), 4),
            "precision_macro": round(float(prec_macro), 4),
            "recall_macro": round(float(rec_macro), 4),
            "f1_macro": round(float(f1_macro), 4),
            "precision_weighted": round(float(prec_w), 4),
            "recall_weighted": round(float(rec_w), 4),
            "f1_weighted": round(float(f1_w), 4),
        },
        "classes": list(le.classes_)
    }
    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

    plot_training_curves(hist_df, curves_base)
    plot_confusion_from_arrays(yte, y_pred, list(le.classes_), plots_dir / "confusion_matrix")
    plot_roc_from_arrays(yte, prob_te, list(le.classes_), plots_dir / "roc_curves")

    print(f"TRAINING TIME - {stem} - {tag}: {t_fit:.3f} s  (effective epochs: {epochs_trained})")
    print(f"Tiempo TESTING      — {stem} — {tag}: {t_test:.3f} s  (n_muestras_test: {len(yte)})")

    with open(metrics_txt, "w", encoding="utf-8") as f:
        f.write(f"Experiment: {stem} — {tag}\n")
        f.write(f"Stratify used: {strat_used}\n")
        f.write(f"Train shape: {Xtr.shape} | Test shape: {Xte.shape}\n")
        f.write("\n== Timings (s) ==\n")
        f.write(f"train_time_s: {t_fit:.6f}\n")
        f.write(f"test_time_s:  {t_test:.6f}\n")
        f.write(f"epochs_trained: {epochs_trained}\n")
        f.write("\n== Metrics ==\n")
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(f"Precision (macro): {prec_macro:.4f}\n")
        f.write(f"Recall (macro): {rec_macro:.4f}\n")
        f.write(f"F1-score (macro): {f1_macro:.4f}\n")
        f.write(f"Precision (weighted): {prec_w:.4f}\n")
        f.write(f"Recall (weighted): {rec_w:.4f}\n")
        f.write(f"F1-score (weighted): {f1_w:.4f}\n")
        f.write("\n== Classification report ==\n")
        f.write(cls_report)

    with open(out_dir / "info.txt", "w", encoding="utf-8") as f:
        f.write(f"Features: {len(feat_cols)}\n")
        f.write("First 10 features:\n")
        f.write(", ".join(feat_cols[:10]) + "\n")
        f.write(f"Classes: {list(le.classes_)}\n")

# ---------------------- IMAGE COLLAGES (NO TITLES) ------------------------

def _letters_seq(n):
    letters = []
    i = 0
    while len(letters) < n:
        s = ""
        k = i
        while True:
            s = chr(ord('A') + (k % 26)) + s
            k = k // 26 - 1
            if k < 0:
                break
        letters.append(s)
        i += 1
    return letters

def _compose_image_grid(*args, **kwargs):
    raise RuntimeError("Use vector collage builders based on saved arrays and histories.")

def _stem_to_dt_label(stem: str) -> str:
    m = re.search(r"ml_win_([0-9]+\.[0-9]{2})s", stem)
    return f"Δt={m.group(1)} s" if m else stem

def _overall_artifacts_for_stem(stem: str):
    base = RESULTS_DIR / stem / "overall" / "plots"
    return {
        "cm": base / "confusion_matrix.pdf",
        "roc": base / "roc_curves.pdf",
        "acc": base / "training_curves_accuracy.pdf",
        "loss": base / "training_curves_loss.pdf",
    }

def build_overall_collages(available_stems):
    return build_overall_vector_collages(RESULTS_DIR, available_stems, OVERALL_COLLAGES_DIR)

def _by_user_artifacts_for_stem_and_user(stem: str, user_id: str):
    base = RESULTS_DIR / stem / "by_user" / f"user_{user_id}" / "plots"
    return {
        "cm": base / "confusion_matrix.pdf",
        "roc": base / "roc_curves.pdf",
        "acc": base / "training_curves_accuracy.pdf",
        "loss": base / "training_curves_loss.pdf",
    }

def build_by_user_collages_for_stem(stem: str):
    return build_by_user_vector_collages_for_stem(RESULTS_DIR, stem)

# ----------------------- MACRO AUC FROM FILES (for tables) -------------------

def _macro_auc_from_files(exp_dir: Path) -> float:
    try:
        with open(exp_dir / "classes.json", "r", encoding="utf-8") as f:
            classes = json.load(f)["classes"]
        y_true = pd.read_csv(exp_dir / "arrays" / "y_true.csv")["y_true"].values
        y_prob = pd.read_csv(exp_dir / "arrays" / "y_prob.csv").values
        n_classes = len(classes)
        y_true_ohe = np.eye(n_classes)[y_true]
        aucs = []
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_ohe[:, i], y_prob[:, i])
            aucs.append(auc(fpr, tpr))
        return float(np.mean(aucs)) if len(aucs) else np.nan
    except Exception:
        return float("nan")

# ------------------------ TABLES (reading metrics.json/arrays) --------------------

def _collect_metrics_rows():
    overall_rows, intra_rows = [], []
    for stem in STEMS:
        stem_dir = RESULTS_DIR / stem
        if not stem_dir.exists():
            continue

        mpath = stem_dir / "overall" / "metrics.json"
        if mpath.exists():
            with open(mpath, "r", encoding="utf-8") as f:
                m = json.load(f)
            auc_macro = _macro_auc_from_files(stem_dir / "overall")
            overall_rows.append({
                "Δt (s)": stem.replace("ml_win_", "").replace("s", ""),
                "Accuracy": m["metrics"]["accuracy"],
                "F1 (macro)": m["metrics"]["f1_macro"],
                "AUC (macro)": auc_macro,
                "stem": stem
            })

        by_user_dir = stem_dir / "by_user"
        if by_user_dir.exists():
            for um in sorted(by_user_dir.glob("user_*")):
                mpath_u = um / "metrics.json"
                if mpath_u.exists():
                    with open(mpath_u, "r", encoding="utf-8") as f:
                        mu = json.load(f)
                    auc_macro_u = _macro_auc_from_files(um)
                    user_id = um.name.replace("user_", "")
                    intra_rows.append({
                        "User": user_id,
                        "Δt (s)": stem.replace("ml_win_", "").replace("s", ""),
                        "Accuracy": mu["metrics"]["accuracy"],
                        "F1 (macro)": mu["metrics"]["f1_macro"],
                        "AUC (macro)": auc_macro_u,
                        "stem": stem
                    })
    return overall_rows, intra_rows

def _print_and_save_tables():
    overall_rows, intra_rows = _collect_metrics_rows()
    if overall_rows:
        df_overall = pd.DataFrame(overall_rows)
        df_overall["Δt (s)"] = df_overall["Δt (s)"].astype(float)
        df_overall = df_overall.sort_values("Δt (s)").reset_index(drop=True)
        for col in ["Accuracy", "F1 (macro)", "AUC (macro)"]:
            df_overall[col] = pd.to_numeric(df_overall[col], errors="coerce").round(4)
        print("\nTable - Overall (80/20): metrics by window size")
        print(df_overall.drop(columns=["stem"]).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
        tables_dir = RESULTS_DIR / "tables"
        ensure_dir(tables_dir)
        df_overall.to_csv(tables_dir / "overall_metrics.csv", index=False)
        with open(tables_dir / "overall_metrics.tex", "w", encoding="utf-8") as f:
            f.write(df_overall.drop(columns=["stem"]).to_latex(index=False, float_format="%.4f"))
    else:
        print("\n[WARN] No metrics OVERALL to build the table.")

    if intra_rows:
        df_intra = pd.DataFrame(intra_rows)
        df_intra["Δt (s)"] = df_intra["Δt (s)"].astype(float)
        df_intra = df_intra.sort_values(["User", "Δt (s)"]).reset_index(drop=True)
        for col in ["Accuracy", "F1 (macro)", "AUC (macro)"]:
            df_intra[col] = pd.to_numeric(df_intra[col], errors="coerce").round(4)
        print("\nTable - Within-user (80/20): metrics by window size")
        def _fmt(x): return f"{x:.4f}" if isinstance(x, float) else str(x)
        print(df_intra.drop(columns=["stem"]).to_string(index=False, formatters={
            "Δt (s)": lambda x: f"{x:.2f}",
            "Accuracy": _fmt, "F1 (macro)": _fmt, "AUC (macro)": _fmt, "User": str
        }))
        tables_dir = RESULTS_DIR / "tables"
        ensure_dir(tables_dir)
        df_intra.to_csv(tables_dir / "intrauser_metrics.csv", index=False)
        with open(tables_dir / "intrauser_metrics.tex", "w", encoding="utf-8") as f:
            f.write(df_intra.drop(columns=["stem"]).to_latex(index=False, float_format="%.4f"))
    else:
        print("\n[WARN] No metrics BY-USER to build the table.")

# --------------------------- RE-RENDER FROM FILES -----------------------------

def _rerender_plots_from_saved_files():
    for stem in STEMS:
        base_over = RESULTS_DIR / stem / "overall"
        if (base_over / "history.csv").exists():
            with open(base_over / "classes.json", "r", encoding="utf-8") as f:
                classes = json.load(f)["classes"]

            hist_df = pd.read_csv(base_over / "history.csv")
            plot_training_curves(hist_df, base_over / "plots" / "training_curves")

            y_true = pd.read_csv(base_over / "arrays" / "y_true.csv")["y_true"].values
            y_pred = pd.read_csv(base_over / "arrays" / "y_pred.csv")["y_pred"].values
            y_prob = pd.read_csv(base_over / "arrays" / "y_prob.csv").values

            plot_confusion_from_arrays(y_true, y_pred, classes, base_over / "plots" / "confusion_matrix")
            plot_roc_from_arrays(y_true, y_prob, classes, base_over / "plots" / "roc_curves")

        by_user_dir = RESULTS_DIR / stem / "by_user"
        if by_user_dir.exists():
            for udir in sorted(by_user_dir.glob("user_*")):
                if not (udir / "history.csv").exists():
                    continue
                with open(udir / "classes.json", "r", encoding="utf-8") as f:
                    classes = json.load(f)["classes"]
                hist_df = pd.read_csv(udir / "history.csv")
                plot_training_curves(hist_df, udir / "plots" / "training_curves")
                y_true = pd.read_csv(udir / "arrays" / "y_true.csv")["y_true"].values
                y_pred = pd.read_csv(udir / "arrays" / "y_pred.csv")["y_pred"].values
                y_prob = pd.read_csv(udir / "arrays" / "y_prob.csv").values
                plot_confusion_from_arrays(y_true, y_pred, classes, udir / "plots" / "confusion_matrix")
                plot_roc_from_arrays(y_true, y_prob, classes, udir / "plots" / "roc_curves")

# ----------------------------- EXECUTION (MAIN) -----------------------------------

def run_all_for_stem(stem: str, df: pd.DataFrame):
    info = basic_info(df, stem)

    stem_dir = RESULTS_DIR / stem
    ensure_dir(stem_dir)
    with open(stem_dir / "dataset_info.txt", "w", encoding="utf-8") as f:
        f.write(info + "\n")

    print(info)

    overall_dir = stem_dir / "overall"
    run_one_experiment(df, stem, overall_dir, tag="overall")

    for user in sorted(df["user"].astype(str).unique().tolist()):
        dfx = df[df["user"].astype(str) == user].reset_index(drop=True)
        user_dir = stem_dir / "by_user" / f"user_{user}"
        run_one_experiment(dfx, stem, user_dir, tag=f"user_{user}")

    build_by_user_collages_for_stem(stem)

def _print_split_sizes(stem: str, tag: str, users_tr, users_te, ytr=None, yte=None, le=None, strat_used=None):
    ntr, nte = len(users_tr), len(users_te)
    total = ntr + nte
    df_summary = pd.DataFrame({"Partition": ["Train", "Test", "Total"], "Rows": [ntr, nte, total]})
    print(f"\n================= SPLIT SIZES — {stem} — {tag} =================")
    print(df_summary.to_string(index=False))
    if strat_used:
        print(f"Stratify usado: {strat_used}")

    if (ytr is not None) and (yte is not None) and (le is not None):
        gtr = pd.Series(le.inverse_transform(ytr)).value_counts().sort_index()
        gte = pd.Series(le.inverse_transform(yte)).value_counts().sort_index()
        df_g = pd.DataFrame({"Train": gtr, "Test": gte}).fillna(0).astype(int)
        print("\nDetail by gesture:")
        print(df_g.to_string())

    if tag == "overall":
        utr = pd.Series(users_tr, dtype=str).value_counts().sort_index()
        ute = pd.Series(users_te, dtype=str).value_counts().sort_index()
        users_all = sorted(set(utr.index).union(set(ute.index)))
        df_usr = pd.DataFrame({
            "User": users_all,
            "Train": [int(utr.get(u, 0)) for u in users_all],
            "Test":  [int(ute.get(u, 0)) for u in users_all],
        })
        df_usr["Total"] = df_usr["Train"] + df_usr["Test"]
        print("\nDetail by user (OVERALL):")
        print(df_usr.to_string(index=False))

        if (yte is not None) and (le is not None):
            gestures_te = pd.Series(le.inverse_transform(yte), index=None, dtype=str)
            ct_te = pd.crosstab(pd.Series(users_te, name="user"), pd.Series(gestures_te, name="gesture"))
            print("\nCrosstab TEST (user x gesture):")
            print(ct_te.to_string())

# ========================== PARTE 1 — ENTRENAR Y GUARDAR ==========================
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    print("Reading datasets from:", DATA_DIR.resolve())
    loaded_any = False
    available_stems = []

    for stem, csv_path in CSV_PATHS.items():
        if not csv_path.exists():
            print(f"[WARN] Not found: {csv_path}")
            continue

        df = pd.read_csv(csv_path, dtype={"user": str, "gesture": str})
        if "user" not in df.columns or "gesture" not in df.columns:
            print(f"[WARN] {stem}: missing columns 'user' o 'gesture'. Skipped.")
            continue

        df = df.sort_values(["user", "gesture"]).reset_index(drop=True)

        run_all_for_stem(stem, df)
        available_stems.append(stem)
        loaded_any = True

    if not loaded_any:
        print("\nNo dataset was loaded. Check the folder:", DATA_DIR.resolve())
    else:
        _rerender_plots_from_saved_files()
        build_overall_collages(available_stems)
        print("\nDone. Artifacts saved to:", RESULTS_DIR.resolve())

# =================== PART 2 - READ AND PLOT FROM FILES ONLY =================
ONLY_RENDER_FROM_FILES = True

if ONLY_RENDER_FROM_FILES:
    _rerender_plots_from_saved_files()

    for stem in STEMS:
        if (RESULTS_DIR / stem).exists():
            build_by_user_collages_for_stem(stem)

    avail = [stem for stem in STEMS if (RESULTS_DIR / stem / "overall" / "arrays" / "y_true.csv").exists()]
    if avail:
        build_overall_collages(avail)

    _print_and_save_tables()

    print("\n[OK] Render and tables ready from files only. Directory:", RESULTS_DIR.resolve())
